# Document Classification
Classify documents:
- UPI trasactions
- Back Integration log
- Complience report
- Partnership SLAs

In [62]:
import numpy as np

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

### Configuration

In [ ]:
MODEL_NAME = "distilbert/distilbert-base-uncased"
DATA_DIR = "../data/fine_tuning/document_classification"
CHECKPOINTS_OUTPUT_DIR = "../data/checkpoints/distilbert-base-uncased-finetuned-document-classification"
MODEL_OUTPUT_DIR = "../data/models/distilbert-base-uncased-finetuned-document-classification"

label2id = {
    "UPI_TRANSACTION": 0,
    "BANK_INTEGRATION_LOG": 1,
    "PARTNERSHIP_AGREEMENT": 2,
    "COMPLIANCE_CIRCULAR": 3
}
id2label = {v: k for k, v in label2id.items()}

### Load dataset

In [64]:
dataset = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/train.jsonl",
        "validation": f"{DATA_DIR}/validation.jsonl",
        "test": f"{DATA_DIR}/test.jsonl"
    }
)

### 3. Convert string labels -> integers

In [65]:
def encode_label(example):
    example["label"] = label2id[example["label"]]
    return example


dataset = dataset.map(encode_label)



### 4. Load tokenizer

In [66]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

### 5. Tokenize documents

In [67]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)


# Dynamic padding
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)



### Load DistilBERT classification model

In [68]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6577.55it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Evaluation metrics

In [69]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


### Training configuration

In [ ]:
training_args = TrainingArguments(
    output_dir=CHECKPOINTS_OUTPUT_DIR,

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=5,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    seed=42,

    report_to="none"
)

### Trainer

In [71]:

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)


### Fine-tune

In [72]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.877042,0.336176,1.000000,1.000000,1.000000,1.000000
2,0.198698,0.072869,1.000000,1.000000,1.000000,1.000000
3,0.056965,0.032101,1.000000,1.000000,1.000000,1.000000
4,0.032058,0.023621,1.000000,1.000000,1.000000,1.000000
5,0.027948,0.021547,1.000000,1.000000,1.000000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.38it/s]
/Users/mangatmurmu/Mangat/payment_document_chatbot/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.52it/s]
/Users/mangatmurmu/Mangat/payment_document_chatbot/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s]
/Users/mangatmurmu/Mangat/payment_document_chatbot/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
W

TrainOutput(global_step=200, training_loss=0.23854226291179656, metrics={'train_runtime': 250.013, 'train_samples_per_second': 3.2, 'train_steps_per_second': 0.8, 'total_flos': 100730560701120.0, 'train_loss': 0.23854226291179656, 'epoch': 5.0})

### Test

In [73]:
test_results = trainer.evaluate(
    tokenized_dataset["test"]
)

print(test_results)


/Users/mangatmurmu/Mangat/payment_document_chatbot/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.027948,0.341891,5,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 0.3418911397457123, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0}


### Save

In [ ]:
trainer.save_model(MODEL_OUTPUT_DIR)

tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.71it/s]


('../data/models/distilbert-base-uncased-finetuned-document-classification/tokenizer_config.json',
 '../data/models/distilbert-base-uncased-finetuned-document-classification/tokenizer.json')

# Named Entity Extraction
Extract
- Circular number
- Issue Date
- Subject

In [75]:
import numpy as np
import evaluate

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)



### Configuration

In [ ]:

MODEL_NAME = "distilbert/distilbert-base-cased"
DATA_DIR = "../data/fine_tuning/named_entity_recognition"
CHECKPOINTS_OUTPUT_DIR = "../data/checkpoints/distilbert-base-cased-finetuned-entity-extraction"
MODEL_OUTPUT_DIR = "../data/models/distilbert-base-cased-finetuned-entity-extraction"

MAX_LENGTH = 512




### BIO labels

In [77]:

label_list = [
    "O",
    "B-DOCUMENT_NUMBER",
    "I-DOCUMENT_NUMBER",
    "B-TITLE",
    "I-TITLE",
    "B-ISSUE_DATE",
    "I-ISSUE_DATE"
]

label2id = {
    label: i
    for i, label in enumerate(label_list)
}

id2label = {
    i: label
    for i, label in enumerate(label_list)
}



### Load dataset

In [78]:

dataset = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/train.jsonl",
        "validation": f"{DATA_DIR}/validation.jsonl",
        "test": f"{DATA_DIR}/test.jsonl"
    }
)



### Tokenizer

In [79]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)




### Character spans -> BIO token labels

In [80]:
def tokenize_and_align_labels(example):

    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True
    )

    offsets = tokenized["offset_mapping"]

    labels = [
        -100 if start == end else label2id["O"]
        for start, end in offsets
    ]


    for entity in example["entities"]:

        entity_start = entity["start"]
        entity_end = entity["end"]
        entity_type = entity["label"]

        first_token = True


        for token_index, (token_start, token_end) in enumerate(offsets):

            if token_start == token_end:
                continue


            overlaps = (
                token_start < entity_end
                and token_end > entity_start
            )


            if overlaps:

                prefix = "B" if first_token else "I"

                labels[token_index] = label2id[
                    f"{prefix}-{entity_type}"
                ]

                first_token = False


    tokenized["labels"] = labels

    tokenized.pop("offset_mapping")

    return tokenized


tokenized_dataset = dataset.map(
    tokenize_and_align_labels
)



Map: 100%|██████████| 10/10 [00:00<00:00, 633.48 examples/s]


### Dynamic padding

In [81]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)



### Model

In [82]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,

    num_labels=len(label_list),

    id2label=id2label,
    label2id=label2id
)



Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6539.70it/s]
[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Metrics

In [83]:
seqeval = evaluate.load("seqeval")


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=2
    )


    true_predictions = []
    true_labels = []


    for prediction, label in zip(
        predictions,
        labels
    ):

        true_predictions.append([
            id2label[p]
            for p, l in zip(prediction, label)
            if l != -100
        ])

        true_labels.append([
            id2label[l]
            for p, l in zip(prediction, label)
            if l != -100
        ])


    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )


    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }



### Training

In [ ]:
training_args = TrainingArguments(
    output_dir=CHECKPOINTS_OUTPUT_DIR,

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=8,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    seed=42,

    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)



### Fine-tune

In [85]:
trainer.train()

/Users/mangatmurmu/Mangat/payment_document_chatbot/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.049077,0.535175,0.000000,0.000000,0.000000,0.904060
2,0.391908,0.334486,0.080000,0.080000,0.080000,0.932030
3,0.291907,0.213212,0.232558,0.400000,0.294118,0.951880
4,0.152936,0.146208,0.400000,0.720000,0.514286,0.959098
5,0.103185,0.097849,0.656250,0.840000,0.736842,0.974737
6,0.070082,0.081998,0.733333,0.880000,0.800000,0.980451
7,0.059321,0.081505,0.733333,0.880000,0.800000,0.980150
8,0.049265,0.077854,0.785714,0.880000,0.830189,0.982256


/Users/mangatmurmu/Mangat/payment_document_chatbot/.venv/lib/python3.13/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/mangatmurmu/Mangat/payment_document_chatbot/.venv/lib/python3.13/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.42it/s]
/Users/mangatmurmu/Mangat/payment_document_chatbot/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init

TrainOutput(global_step=104, training_loss=0.2709599263392962, metrics={'train_runtime': 302.1549, 'train_samples_per_second': 1.324, 'train_steps_per_second': 0.344, 'total_flos': 47625318738096.0, 'train_loss': 0.2709599263392962, 'epoch': 8.0})

### Final test evaluation

In [86]:
test_results = trainer.evaluate(
    tokenized_dataset["test"]
)

print(test_results)



/Users/mangatmurmu/Mangat/payment_document_chatbot/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.049265,0.035232,8,0.933333,0.965517,0.949153,0.992481


{'eval_loss': 0.03523151949048042, 'eval_precision': 0.9333333333333333, 'eval_recall': 0.9655172413793104, 'eval_f1': 0.9491525423728815, 'eval_accuracy': 0.9924812030075187}


### Save model

In [ ]:
trainer.save_model(MODEL_OUTPUT_DIR)

tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.32s/it]


('../data/models/distilbert-base-cased-finetuned-entity-extraction/tokenizer_config.json',
 '../data/models/distilbert-base-cased-finetuned-entity-extraction/tokenizer.json')